# 06 — Implantação

**CRISP-DM fase 6.** Publica a segmentação PeopleCluster (Tema 08 · RH),
define contrato de classificação, monitoramento e entregas operacionais.

**Modelo publicado:** K-Medoids / Gower, k = 2  
**Entrada:** `rotulos_clusters.csv`, `matriz_gower.npy`, `catalogo_personas.json`  
**Saída:** `models/pacote_implantacao.json`

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config
from src.deployment import scoring

config.garantir_diretorios()
FIG = config.FIGURES
DOC_FIG = config.DOCS_FIGURES

## 0. Licença para implantar

A fase 5 autorizou seguir com a partição principal apesar da silhueta baixa,
porque o contraste de attrition (~11% vs ~21%) sustenta ação de RH.

In [ ]:
features = pd.read_csv(config.DATA_PROCESSED / "hr_features_cluster.csv")
avaliacao = pd.read_csv(config.DATA_PROCESSED / "hr_avaliacao.csv")
rotulos_df = pd.read_csv(config.DATA_PROCESSED / "rotulos_clusters.csv")
gower = np.load(config.DATA_PROCESSED / "matriz_gower.npy")
decisao = json.loads((config.MODELS / "decisao_modelagem.json").read_text(encoding="utf-8"))
catalogo = json.loads((config.MODELS / "catalogo_personas.json").read_text(encoding="utf-8"))
criterios = pd.read_csv(config.TABLES / "avaliacao_criterios.csv")

rotulos = rotulos_df["cluster_kmedoids_gower"].to_numpy()
assert int(decisao["k"]) == 2
assert len(features) == len(rotulos) == 1470

display(criterios)
pd.DataFrame(catalogo)

## 1. Plano de implantação

| Camada | Conteúdo |
|---|---|
| Pacote | medoides Gower + catálogo de ações |
| Classificação | colaborador novo → cluster do medoide mais próximo |
| Entrega | CSV classificado, app Streamlit, KPIs para RH |
| Monitoramento | attrition, satisfação e overtime por segmento |

In [ ]:
pacote = scoring.publicar_pacote(features, rotulos, avaliacao, gower)
{
    "versao": pacote["versao"],
    "k": pacote["k"],
    "medoides_employee_number": pacote["medoides_employee_number"],
    "attrition_por_cluster": pacote["referencia"]["attrition_por_cluster"],
}

## 1.1 Contrato de entrada

O classificador exige as 29 features de `hr_features_cluster`.
Atributos ausentes → rejeição explícita (não imputa em silêncio).

In [ ]:
# Ensaio válido: usa um registro real da base
exemplo_ok = features.iloc[0].to_dict()
resultado_ok = scoring.classificar_colaborador(exemplo_ok, pacote=pacote)

# Ensaio inválido: remove campos obrigatórios
exemplo_ruim = {k: v for k, v in exemplo_ok.items() if k not in {"MonthlyIncome", "OverTime"}}
resultado_ruim = scoring.classificar_colaborador(exemplo_ruim, pacote=pacote)

pd.DataFrame(
    [
        {
            "caso": "registro completo",
            **{k: resultado_ok.get(k) for k in ["ok", "cluster", "confianca", "persona"]},
        },
        {"caso": "faltam atributos", **{k: resultado_ruim.get(k) for k in ["ok", "erros"]}},
    ]
)

## 1.2 Reprodução da carteira e confiança

In [ ]:
# Distâncias aos medoides para toda a base (via Gower já calculada)
medoides_idx = np.array(pacote["medoides_idx"], dtype=int)
dist_med = gower[:, medoides_idx]
pred = dist_med.argmin(axis=1)
acordancia = float((pred == rotulos).mean())

ordenado = np.sort(dist_med, axis=1)
margem = ordenado[:, 1] - ordenado[:, 0]
faixa = np.where(margem >= 0.02, "alta", np.where(margem >= 0.01, "media", "baixa"))

carteira = avaliacao[["EmployeeNumber", "Attrition"]].copy()
carteira["cluster"] = rotulos
carteira["cluster_reclassificado"] = pred
carteira["confianca"] = faixa
carteira["persona"] = carteira["cluster"].map(
    {int(item["cluster"]): item["persona"] for item in catalogo}
)
carteira.to_csv(config.DATA_PROCESSED / "carteira_classificada.csv", index=False)

print(f"Concordância medoide↔rótulo publicado: {acordancia:.2%}")
carteira["confianca"].value_counts(normalize=True).mul(100).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
carteira.groupby("cluster")["Attrition"].apply(lambda s: (s == "Yes").mean() * 100).plot(
    kind="bar", ax=axes[0], color=["#74C69D", "#EE6C4D"], rot=0
)
axes[0].set_title("Attrition por segmento publicado")
axes[0].set_ylabel("%")

carteira["confianca"].value_counts().reindex(["alta", "media", "baixa"]).plot(
    kind="bar", ax=axes[1], color="#1B4965", rot=0
)
axes[1].set_title("Faixa de confiança da atribuição")
plt.tight_layout()
fig.savefig(FIG / "13_implantacao_segmentos.png", dpi=120, bbox_inches="tight")
fig.savefig(DOC_FIG / "13_implantacao_segmentos.png", dpi=120, bbox_inches="tight")
plt.show()

## 1.3 Personas e entregas

In [ ]:
entregas = pd.DataFrame(
    [
        {"entrega": "carteira_classificada.csv", "consumidor": "RH / People Analytics"},
        {"entrega": "pacote_implantacao.json", "consumidor": "TI / scoring"},
        {"entrega": "catalogo_personas.json", "consumidor": "Gestores / RHBP"},
        {"entrega": "App Streamlit", "consumidor": "consulta rápida de personas"},
        {"entrega": "Pitch 10 min", "consumidor": "Diretoria"},
    ]
)
entregas.to_csv(config.TABLES / "implantacao_entregas.csv", index=False)

janelas = pd.DataFrame(
    [
        {"janela": "Sombra", "periodo": "D+0 a D+30", "acao": "classificar sem acionar política"},
        {
            "janela": "Piloto cluster 1",
            "periodo": "D+30 a D+90",
            "acao": "mentoria + revisão salarial amostral",
        },
        {
            "janela": "Rollout",
            "periodo": "D+90+",
            "acao": "ações dos dois segmentos + KPIs mensais",
        },
    ]
)
display(entregas)
janelas

## 2. Monitoramento e manutenção

In [ ]:
plano = pd.DataFrame(
    [
        {
            "indicador": "Attrition por cluster",
            "limiar": "cluster 1 > 25% em janela móvel 90 dias",
            "responsavel": "RHBP",
            "acao": "revisar pacote de retenção do segmento",
        },
        {
            "indicador": "JobSatisfaction média",
            "limiar": "queda > 0,3 pontos vs baseline do segmento",
            "responsavel": "Clima organizacional",
            "acao": "pesquisa dirigida + PDI",
        },
        {
            "indicador": "% OverTime",
            "limiar": "cluster 1 > 40%",
            "responsavel": "Gestão de pessoas",
            "acao": "prevenção de burnout / redistribuição de carga",
        },
        {
            "indicador": "% atribuições de confiança baixa",
            "limiar": "> 20% das novas contratações",
            "responsavel": "People Analytics",
            "acao": "revisar medoides / retreinar",
        },
        {
            "indicador": "Reciclagem do modelo",
            "limiar": "a cada 12 meses ou mudança estrutural de headcount",
            "responsavel": "People Analytics",
            "acao": "reexecutar notebooks 03–05 e republicar pacote",
        },
    ]
)
plano.to_csv(config.TABLES / "implantacao_monitoramento.csv", index=False)
plano

## 3. Relatório final e pitch

In [ ]:
resultados_chave = pd.DataFrame(
    [
        {"pergunta": "Quantos grupos?", "resposta": "2 (K-Medoids/Gower)"},
        {
            "pergunta": "O que caracteriza cada grupo?",
            "resposta": "0 estáveis (attrition ~11%); 1 risco/início de carreira (~21%)",
        },
        {
            "pergunta": "Coocorrências / contraste",
            "resposta": "Renda e tempo de casa menores no grupo de maior saída",
        },
        {
            "pergunta": "Ação diferenciada",
            "resposta": "Retenção no 0; salário/clima/mentoria no 1",
        },
    ]
)

roteiro_pitch = pd.DataFrame(
    [
        {"min": "0-2", "slide": "Problema: política única de RH"},
        {"min": "2-4", "slide": "Método: Gower + K-Medoids, k≤5"},
        {"min": "4-7", "slide": "Dois perfis e contraste de attrition"},
        {"min": "7-9", "slide": "Ações + KPIs de monitoramento"},
        {"min": "9-10", "slide": "Limitações (base sintética) e próximos passos"},
    ]
)

licoes = pd.DataFrame(
    [
        {"licao": "Reservar Attrition/Performance evita vazamento na clusterização"},
        {"licao": "Silhueta baixa não impede valor gerencial se o contraste de negócio for claro"},
        {
            "licao": (
                "Métodos divergem (ARI baixo): publicar um método principal "
                "e documentar comparativos"
            )
        },
        {"licao": "Implantação precisa de contrato de entrada e faixa de confiança"},
    ]
)

resultados_chave.to_csv(config.TABLES / "implantacao_perguntas_diretoria.csv", index=False)
roteiro_pitch.to_csv(config.TABLES / "implantacao_pitch.csv", index=False)
licoes.to_csv(config.TABLES / "implantacao_licoes.csv", index=False)

display(resultados_chave)
display(roteiro_pitch)
licoes

In [ ]:
metadados = {
    "fase": 6,
    "pacote": "models/pacote_implantacao.json",
    "carteira": "data/processed/carteira_classificada.csv",
    "app": "uv run invoke app",
    "concordancia_medoides": acordancia,
    "n_confianca_baixa": int((carteira["confianca"] == "baixa").sum()),
}
(config.MODELS / "metadados_implantacao.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False), encoding="utf-8"
)
metadados

### Encerramento da fase 6

A segmentação está **implantável**: pacote versionado, classificação com rejeição de entrada inválida,
carteira classificada, plano de monitoramento e roteiro de pitch.

Para consultar personas: `uv run invoke app`.